In [1]:
# ============================================================
# ICDM Extra Experimental Series for Fairness-Aware Synthetic Data
# CPU-only, parallel, resumable
#
# Series:
#   1. Intersectional stress test
#   2. Minimum group support sensitivity
#   3. Lambda sweep / Pareto
#   4. CTGAN / TVAE baselines via SDV, optional
#   5. Synthetic quality vs fairness aggregation
#   6. Runtime scaling
#
# Expected extra runs/evaluations:
#   Lambda sweep:
#       4 datasets × 1 setting × 7 lambdas × 5 seeds = 140
#   Intersectional stress:
#       4 datasets × several sensitive configs × 3 methods × 5 seeds
#   Runtime scaling:
#       5 sizes × 2 settings × 3 methods × 3 seeds = 90
#   CTGAN/TVAE optional:
#       4 datasets × 3 settings × 2 methods × 3 seeds = 72
#   Group support:
#       re-evaluation from stored predictions is not available,
#       so it reruns selected configs with different MIN_GROUP_COUNT.
#
# ============================================================

import os

# ------------------------------------------------------------
# CPU-only / avoid oversubscription
# ------------------------------------------------------------

os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import json
import time
import math
import shutil
import traceback
import warnings
import subprocess
import sys
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
)
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


# ============================================================
# 0. Paths and global config
# ============================================================

BASE = Path(os.environ.get("A100", "/home/tahiti/DataGenaration")).expanduser().resolve()
DATA_DIR = BASE / "fairness_datasets"
PROCESSED_DIR = DATA_DIR / "processed"

PREV_RESULTS_DIR = BASE / "RESULTS" / "icdm_fairness_final_cpu"
PREV_ALL_RESULTS = PREV_RESULTS_DIR / "all_results.csv"

RESULTS_DIR = BASE / "RESULTS" / "icdm_fairness_extra_series_cpu"
RUNS_DIR = RESULTS_DIR / "runs_json"

SERIES_DIRS = {
    "lambda_sweep": RUNS_DIR / "lambda_sweep",
    "support_sensitivity": RUNS_DIR / "support_sensitivity",
    "intersectional_stress": RUNS_DIR / "intersectional_stress",
    "sdv_baselines": RUNS_DIR / "sdv_baselines",
    "runtime_scaling": RUNS_DIR / "runtime_scaling",
}

for p in [RESULTS_DIR, RUNS_DIR] + list(SERIES_DIRS.values()):
    p.mkdir(parents=True, exist_ok=True)

# CPU workers. Keep modest because sklearn/SDV may be heavy.
MAX_WORKERS = max(1, min(10, (os.cpu_count() or 4) // 2))

# Main seeds.
SEEDS_5 = [0, 1, 2, 3, 4]
SEEDS_3 = [0, 1, 2]

# If SDV is installed, CTGAN/TVAE can run.
# Set to False if you want to skip neural baselines.
ENABLE_SDV_BASELINES = True

# Drop sensitive columns from downstream classifier.
DROP_CURRENT_SENSITIVE_FROM_MODEL = True

# Synthetic train size.
SYNTH_SIZE_RATIO = 1.0

# Default fairness support.
DEFAULT_MIN_GROUP_COUNT = 20


# ============================================================
# 1. Dataset registry
# ============================================================

DATASET_FILES = {
    "adult": "adult_processed.csv",
    "compas": "compas_processed.csv",
    "german_credit": "german_credit_processed.csv",
    "bank_marketing": "bank_marketing_processed.csv",
    "default_credit_card": "default_credit_card_processed.csv",
    "communities_crime": "communities_crime_processed.csv",
    "student_performance": "student_performance_processed.csv",
    "heart_disease": "heart_disease_processed.csv",
}

MAIN_DATASETS = [
    "adult",
    "compas",
    "german_credit",
    "bank_marketing",
    "default_credit_card",
    "communities_crime",
]

STRESS_DATASETS = [
    "adult",
    "compas",
    "default_credit_card",
    "bank_marketing",
]

LAMBDA_DATASETS = [
    "adult",
    "compas",
    "default_credit_card",
    "bank_marketing",
]

SUPPORT_DATASETS = [
    "adult",
    "compas",
    "default_credit_card",
]

SDV_DATASETS = [
    "adult",
    "compas",
    "default_credit_card",
    "bank_marketing",
]

RUNTIME_DATASETS = [
    "adult",
]

SENSITIVE_SETTINGS = {
    "adult": {
        "single_binary": ["sex"],
        "single_nonbinary": ["race"],
        "age_binned": ["age_group"],
        "intersectional": ["sex", "race", "age_group"],
    },
    "compas": {
        "single_binary": ["sex"],
        "single_nonbinary": ["race"],
        "age_binned": ["age_cat"],
        "intersectional": ["sex", "race", "age_cat"],
    },
    "german_credit": {
        "single_binary": ["foreign_worker"],
        "single_nonbinary": ["personal_status_sex"],
        "age_binned": ["age_group"],
        "intersectional": ["personal_status_sex", "foreign_worker", "age_group"],
    },
    "bank_marketing": {
        "single_binary": ["marital_binary"],
        "single_nonbinary": ["education"],
        "age_binned": ["age_group"],
        "intersectional": ["marital", "education", "age_group"],
    },
    "default_credit_card": {
        "single_binary": ["SEX"],
        "single_nonbinary": ["EDUCATION"],
        "age_binned": ["age_group"],
        "intersectional": ["SEX", "EDUCATION", "MARRIAGE", "age_group"],
    },
    "communities_crime": {
        "single_binary": ["black_high"],
        "single_nonbinary": ["black_group"],
        "age_binned": ["young_group"],
        "intersectional": ["black_group", "white_group", "hisp_group"],
    },
    "student_performance": {
        "single_binary": ["sex"],
        "single_nonbinary": ["address"],
        "age_binned": ["age_group"],
        "intersectional": ["sex", "address", "famsize", "age_group"],
    },
    "heart_disease": {
        "single_binary": ["sex"],
        "single_nonbinary": ["age_group"],
        "age_binned": ["age_group"],
        "intersectional": ["sex", "age_group"],
    },
}

# Stress-test configs: increasing intersectional complexity.
STRESS_SENSITIVE_CONFIGS = {
    "adult": {
        "stress_1_sex": ["sex"],
        "stress_2_sex_race": ["sex", "race"],
        "stress_3_sex_race_age": ["sex", "race", "age_group"],
        "stress_4_sex_race_age_education": ["sex", "race", "age_group", "education"],
    },
    "compas": {
        "stress_1_sex": ["sex"],
        "stress_2_sex_race": ["sex", "race"],
        "stress_3_sex_race_age": ["sex", "race", "age_cat"],
    },
    "default_credit_card": {
        "stress_1_sex": ["SEX"],
        "stress_2_sex_education": ["SEX", "EDUCATION"],
        "stress_3_sex_education_age": ["SEX", "EDUCATION", "age_group"],
        "stress_4_sex_education_marriage_age": ["SEX", "EDUCATION", "MARRIAGE", "age_group"],
    },
    "bank_marketing": {
        "stress_1_marital": ["marital_binary"],
        "stress_2_marital_education": ["marital", "education"],
        "stress_3_marital_education_age": ["marital", "education", "age_group"],
        "stress_4_marital_education_job_age": ["marital", "education", "job", "age_group"],
    },
}

STRESS_METHODS = [
    "vanilla_gdt",
    "single_fair_gdt",
    "our_multi_fair_gdt",
]

LAMBDA_VALUES = [
    0.0,
    0.1,
    0.25,
    0.5,
    1.0,
    2.0,
    4.0,
]

SUPPORT_THRESHOLDS = [
    10,
    20,
    50,
    100,
]

RUNTIME_TRAIN_SIZES = [
    2000,
    5000,
    10000,
    20000,
    "full",
]

RUNTIME_SETTINGS = [
    "single_binary",
    "intersectional",
]

RUNTIME_METHODS = [
    "vanilla_gdt",
    "single_fair_gdt",
    "our_multi_fair_gdt",
]

SDV_METHODS = [
    "ctgan",
    "tvae",
]

SDV_SETTINGS = [
    "single_binary",
    "single_nonbinary",
    "intersectional",
]


# ============================================================
# 2. General helpers
# ============================================================

def log(msg):
    print(msg, flush=True)


def safe_float(x):
    try:
        if x is None:
            return None
        if isinstance(x, (np.floating, np.integer)):
            return float(x)
        if isinstance(x, float) and (math.isnan(x) or math.isinf(x)):
            return None
        return float(x)
    except Exception:
        return None


def qcut_safe(s, q=4):
    s = pd.to_numeric(s, errors="coerce")
    try:
        return pd.qcut(s, q=q, labels=False, duplicates="drop").astype("float")
    except Exception:
        return pd.Series(np.zeros(len(s)), index=s.index, dtype="float")


def make_group_key(df, sensitive_cols):
    if len(sensitive_cols) == 0:
        return pd.Series(["ALL"] * len(df), index=df.index)

    parts = []
    for c in sensitive_cols:
        if c not in df.columns:
            parts.append(pd.Series(["MISSING"] * len(df), index=df.index))
        else:
            parts.append(df[c].astype(str).fillna("NA"))

    key = parts[0]
    for p in parts[1:]:
        key = key + "||" + p
    return key


def clean_dataframe(df):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    df = df.dropna(axis=1, how="all")
    df = df.replace([np.inf, -np.inf], np.nan)

    if "target" not in df.columns:
        raise ValueError("Dataset has no target column")

    df = df.dropna(subset=["target"]).copy()

    y = pd.to_numeric(df["target"], errors="coerce")
    if y.notna().mean() > 0.95:
        df["target"] = y.fillna(0).astype(int)
    else:
        vals = sorted(df["target"].dropna().unique().tolist())
        mapping = {v: i for i, v in enumerate(vals)}
        df["target"] = df["target"].map(mapping).fillna(0).astype(int)

    if df["target"].nunique() > 2:
        df["target"] = (df["target"] > 0).astype(int)

    return df


def add_derived_columns(df, dataset_name):
    df = df.copy()

    if "age_group" not in df.columns:
        for age_col in ["age", "AGE", "AGEP"]:
            if age_col in df.columns:
                df["age_group"] = qcut_safe(df[age_col], q=4)
                break

    if dataset_name == "bank_marketing":
        if "marital" in df.columns:
            df["marital_binary"] = np.where(
                df["marital"].astype(str).str.lower().eq("married"),
                "married",
                "not_married",
            )

    if dataset_name == "communities_crime":
        if "racepctblack" in df.columns:
            x = pd.to_numeric(df["racepctblack"], errors="coerce")
            df["black_high"] = np.where(
                x > x.median(),
                "high_black_share",
                "low_black_share",
            )
            if "black_group" not in df.columns:
                df["black_group"] = qcut_safe(x, q=4)

        if "racePctWhite" in df.columns and "white_group" not in df.columns:
            df["white_group"] = qcut_safe(df["racePctWhite"], q=4)

        if "racePctHisp" in df.columns and "hisp_group" not in df.columns:
            df["hisp_group"] = qcut_safe(df["racePctHisp"], q=4)

        if "racePctAsian" in df.columns and "asian_group" not in df.columns:
            df["asian_group"] = qcut_safe(df["racePctAsian"], q=4)

        if "agePct12t29" in df.columns:
            df["young_group"] = qcut_safe(df["agePct12t29"], q=4)
        elif "agePct16t24" in df.columns:
            df["young_group"] = qcut_safe(df["agePct16t24"], q=4)

    return df


def load_dataset(dataset_name):
    path = PROCESSED_DIR / DATASET_FILES[dataset_name]
    if not path.exists():
        raise FileNotFoundError(path)

    df = pd.read_csv(path)
    df = clean_dataframe(df)
    df = add_derived_columns(df, dataset_name)
    return df


def get_feature_columns(df, sensitive_cols):
    cols = [c for c in df.columns if c != "target"]

    if DROP_CURRENT_SENSITIVE_FROM_MODEL:
        cols = [c for c in cols if c not in sensitive_cols]

    # Avoid leakage.
    if "ViolentCrimesPerPop" in cols:
        cols.remove("ViolentCrimesPerPop")

    if "G3" in cols:
        cols.remove("G3")

    return cols


def split_columns(df, feature_cols):
    numeric_cols = []
    categorical_cols = []

    for c in feature_cols:
        if c not in df.columns:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            numeric_cols.append(c)
        else:
            categorical_cols.append(c)

    return numeric_cols, categorical_cols


def make_preprocessor(df, feature_cols):
    numeric_cols, categorical_cols = split_columns(df, feature_cols)

    try:
        onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        onehot = OneHotEncoder(handle_unknown="ignore", sparse=True)

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=False)),
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", onehot),
    ])

    transformers = []
    if numeric_cols:
        transformers.append(("num", numeric_pipe, numeric_cols))
    if categorical_cols:
        transformers.append(("cat", categorical_pipe, categorical_cols))

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        sparse_threshold=0.3,
    )


def make_downstream_models(seed):
    return {
        "logreg": LogisticRegression(
            max_iter=500,
            solver="liblinear",
            class_weight="balanced",
            random_state=seed,
        ),
        "rf": RandomForestClassifier(
            n_estimators=120,
            max_depth=None,
            min_samples_leaf=3,
            n_jobs=1,
            class_weight="balanced_subsample",
            random_state=seed,
        ),
    }


# ============================================================
# 3. Fairness metrics
# ============================================================

def positive_rate(y_pred):
    y_pred = np.asarray(y_pred)
    if len(y_pred) == 0:
        return np.nan
    return float(np.mean(y_pred == 1))


def safe_rate(mask, values):
    mask = np.asarray(mask)
    values = np.asarray(values)
    if mask.sum() == 0:
        return np.nan
    return float(np.mean(values[mask] == 1))


def compute_fairness_metrics(df_eval, y_true, y_pred, sensitive_cols, min_group_count=20):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    groups = make_group_key(df_eval, sensitive_cols).astype(str)
    group_counts = groups.value_counts()
    valid_groups = group_counts[group_counts >= min_group_count].index.tolist()

    out = {
        "n_groups_total": int(group_counts.shape[0]),
        "n_groups_valid": int(len(valid_groups)),
        "min_group_count": int(group_counts.min()) if len(group_counts) else 0,
        "max_group_count": int(group_counts.max()) if len(group_counts) else 0,
        "support_threshold": int(min_group_count),
    }

    if len(valid_groups) <= 1:
        out.update({
            "dp_diff": None,
            "dp_max_gap": None,
            "equal_opp_diff": None,
            "equalized_odds_diff": None,
            "avg_odds_diff": None,
            "worst_group_positive_rate": None,
            "best_group_positive_rate": None,
        })
        return out

    global_pr = positive_rate(y_pred)
    global_tpr = safe_rate(y_true == 1, y_pred)
    global_fpr = safe_rate(y_true == 0, y_pred)

    group_prs = []
    tpr_gaps = []
    fpr_gaps = []
    eq_odds_gaps = []

    for g in valid_groups:
        m = groups.values == g

        pr_g = positive_rate(y_pred[m])
        if not np.isnan(pr_g):
            group_prs.append(pr_g)

        tpr_g = safe_rate(m & (y_true == 1), y_pred)
        fpr_g = safe_rate(m & (y_true == 0), y_pred)

        local = []

        if not np.isnan(tpr_g) and not np.isnan(global_tpr):
            gap = abs(tpr_g - global_tpr)
            tpr_gaps.append(gap)
            local.append(gap)

        if not np.isnan(fpr_g) and not np.isnan(global_fpr):
            gap = abs(fpr_g - global_fpr)
            fpr_gaps.append(gap)
            local.append(gap)

        if local:
            eq_odds_gaps.append(max(local))

    dp_abs = [abs(x - global_pr) for x in group_prs]
    dp_diff = max(dp_abs) if dp_abs else None
    dp_max_gap = max(group_prs) - min(group_prs) if group_prs else None

    equal_opp_diff = max(tpr_gaps) if tpr_gaps else None
    fpr_diff = max(fpr_gaps) if fpr_gaps else None
    equalized_odds_diff = max(eq_odds_gaps) if eq_odds_gaps else None

    if equal_opp_diff is not None and fpr_diff is not None:
        avg_odds_diff = 0.5 * (equal_opp_diff + fpr_diff)
    else:
        avg_odds_diff = None

    out.update({
        "dp_diff": safe_float(dp_diff),
        "dp_max_gap": safe_float(dp_max_gap),
        "equal_opp_diff": safe_float(equal_opp_diff),
        "equalized_odds_diff": safe_float(equalized_odds_diff),
        "avg_odds_diff": safe_float(avg_odds_diff),
        "worst_group_positive_rate": safe_float(max(group_prs) if group_prs else None),
        "best_group_positive_rate": safe_float(min(group_prs) if group_prs else None),
    })

    return out


# ============================================================
# 4. Synthetic quality metrics
# ============================================================

def categorical_tvd(real_s, synth_s):
    r = real_s.astype(str).fillna("NA")
    s = synth_s.astype(str).fillna("NA")

    cats = sorted(set(r.unique()).union(set(s.unique())))
    if len(cats) == 0:
        return None

    rp = r.value_counts(normalize=True).reindex(cats).fillna(0.0)
    sp = s.value_counts(normalize=True).reindex(cats).fillna(0.0)

    return float(0.5 * np.abs(rp.values - sp.values).sum())


def numeric_ks_like(real_s, synth_s):
    r = pd.to_numeric(real_s, errors="coerce").dropna().values
    s = pd.to_numeric(synth_s, errors="coerce").dropna().values

    if len(r) < 2 or len(s) < 2:
        return None

    values = np.sort(np.unique(np.concatenate([r, s])))
    if len(values) > 1000:
        rng = np.random.default_rng(123)
        values = np.sort(rng.choice(values, size=1000, replace=False))

    r_sorted = np.sort(r)
    s_sorted = np.sort(s)

    r_cdf = np.searchsorted(r_sorted, values, side="right") / len(r_sorted)
    s_cdf = np.searchsorted(s_sorted, values, side="right") / len(s_sorted)

    return float(np.max(np.abs(r_cdf - s_cdf)))


def correlation_distance(real_df, synth_df, numeric_cols):
    cols = [c for c in numeric_cols if c in real_df.columns and c in synth_df.columns]
    if len(cols) < 2:
        return None

    r = real_df[cols].apply(pd.to_numeric, errors="coerce")
    s = synth_df[cols].apply(pd.to_numeric, errors="coerce")

    r = r.fillna(r.median(numeric_only=True)).fillna(0.0)
    s = s.fillna(s.median(numeric_only=True)).fillna(0.0)

    try:
        rc = r.corr().fillna(0.0).values
        sc = s.corr().fillna(0.0).values
        return float(np.mean(np.abs(rc - sc)))
    except Exception:
        return None


def detection_auc(real_df, synth_df, feature_cols, seed):
    n = min(len(real_df), len(synth_df), 5000)
    if n < 50:
        return None

    real_sample = real_df.sample(n=n, random_state=seed)
    synth_sample = synth_df.sample(n=n, random_state=seed)

    a = real_sample[feature_cols].copy()
    b = synth_sample[feature_cols].copy()

    a["_is_synth"] = 0
    b["_is_synth"] = 1

    combined = pd.concat([a, b], axis=0, ignore_index=True)
    y = combined["_is_synth"].astype(int).values
    X = combined.drop(columns=["_is_synth"])

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.35,
        random_state=seed,
        stratify=y,
    )

    pre = make_preprocessor(X_train, list(X_train.columns))

    clf = Pipeline([
        ("pre", pre),
        ("clf", LogisticRegression(max_iter=300, solver="liblinear", random_state=seed)),
    ])

    try:
        clf.fit(X_train, y_train)
        proba = clf.predict_proba(X_test)[:, 1]
        return float(roc_auc_score(y_test, proba))
    except Exception:
        return None


def compute_quality_metrics(real_train, synth_train, feature_cols, seed):
    numeric_cols, categorical_cols = split_columns(real_train, feature_cols)

    ks_vals = []
    for c in numeric_cols:
        if c in synth_train.columns:
            v = numeric_ks_like(real_train[c], synth_train[c])
            if v is not None:
                ks_vals.append(v)

    tvd_vals = []
    for c in categorical_cols:
        if c in synth_train.columns:
            v = categorical_tvd(real_train[c], synth_train[c])
            if v is not None:
                tvd_vals.append(v)

    corr_dist = correlation_distance(real_train, synth_train, numeric_cols)
    det_auc = detection_auc(real_train, synth_train, feature_cols, seed)

    return {
        "quality_numeric_ks_mean": safe_float(np.mean(ks_vals) if ks_vals else None),
        "quality_categorical_tvd_mean": safe_float(np.mean(tvd_vals) if tvd_vals else None),
        "quality_corr_distance": safe_float(corr_dist),
        "quality_detection_auc": safe_float(det_auc),
    }


# ============================================================
# 5. Synthetic generators
# ============================================================

def repair_synthetic_dtypes(real_df, synth_df):
    synth = synth_df.copy()

    for c in real_df.columns:
        if c not in synth.columns:
            synth[c] = real_df[c].sample(n=len(synth), replace=True, random_state=0).values

    synth = synth[real_df.columns]

    for c in real_df.columns:
        if c == "target":
            synth[c] = pd.to_numeric(synth[c], errors="coerce").round().fillna(0).astype(int)
            synth[c] = np.where(synth[c] > 0, 1, 0)
            continue

        if pd.api.types.is_numeric_dtype(real_df[c]):
            synth[c] = pd.to_numeric(synth[c], errors="coerce")

            med = pd.to_numeric(real_df[c], errors="coerce").median()
            if pd.isna(med):
                med = 0.0

            synth[c] = synth[c].fillna(med)

            real_nonnull = pd.to_numeric(real_df[c], errors="coerce").dropna()
            if len(real_nonnull) > 0:
                is_int_like = np.allclose(real_nonnull.values, np.round(real_nonnull.values), atol=1e-8)
                if is_int_like:
                    synth[c] = np.round(synth[c]).astype(int)
        else:
            mode = real_df[c].mode(dropna=True)
            fill = mode.iloc[0] if len(mode) else "NA"
            synth[c] = synth[c].astype(object).where(~synth[c].isna(), fill).astype(str)

    return synth


def generate_bootstrap(train_df, n_samples, seed):
    return train_df.sample(n=n_samples, replace=True, random_state=seed).reset_index(drop=True)


def build_tree_leaf_buckets(train_df, seed, n_estimators=48, min_samples_leaf=8):
    feature_cols = [c for c in train_df.columns if c != "target"]
    X = train_df[feature_cols]
    y = train_df["target"].astype(int).values

    pre = make_preprocessor(train_df, feature_cols)

    forest = ExtraTreesClassifier(
        n_estimators=n_estimators,
        max_depth=None,
        min_samples_leaf=min_samples_leaf,
        n_jobs=1,
        random_state=seed,
        class_weight=None,
    )

    pipe = Pipeline([
        ("pre", pre),
        ("forest", forest),
    ])

    pipe.fit(X, y)

    X_enc = pipe.named_steps["pre"].transform(X)
    leaves = pipe.named_steps["forest"].apply(X_enc)

    buckets = {}
    n_trees = leaves.shape[1]

    for t in range(n_trees):
        for leaf_id in np.unique(leaves[:, t]):
            idx = np.where(leaves[:, t] == leaf_id)[0]
            if len(idx) > 0:
                buckets[(t, int(leaf_id))] = idx

    keys = list(buckets.keys())
    sizes = np.array([len(buckets[k]) for k in keys], dtype=float)
    sizes = sizes / sizes.sum()

    return {
        "keys": keys,
        "sizes": sizes,
        "buckets": buckets,
    }


def compute_row_fair_weights(
    train_df,
    sensitive_cols,
    mode="multi_intersection",
    alpha=2.0,
    lambda_fair=1.0,
    cap=30.0,
):
    n = len(train_df)
    y = train_df["target"].astype(int).values
    weights = np.ones(n, dtype=float)

    if mode == "none" or not sensitive_cols:
        return weights / weights.sum()

    global_counts = pd.Series(y).value_counts().reindex([0, 1]).fillna(0.0).values + alpha
    global_probs = global_counts / global_counts.sum()

    if mode == "single_only":
        sensitive_sets = [[sensitive_cols[0]]]

    elif mode == "separate_only":
        sensitive_sets = [[c] for c in sensitive_cols]

    elif mode == "multi_intersection":
        sensitive_sets = [[c] for c in sensitive_cols]
        if len(sensitive_cols) > 1:
            sensitive_sets.append(list(sensitive_cols))

    else:
        sensitive_sets = [list(sensitive_cols)]

    for sens_set in sensitive_sets:
        group_key = make_group_key(train_df, sens_set)

        tmp = pd.DataFrame({
            "group": group_key.values,
            "target": y,
        })

        group_target_counts = (
            tmp.groupby(["group", "target"])
            .size()
            .unstack(fill_value=0)
            .reindex(columns=[0, 1], fill_value=0)
        )

        group_target_counts = group_target_counts[[0, 1]].astype(float) + alpha
        group_probs = group_target_counts.div(group_target_counts.sum(axis=1), axis=0)

        ratio = {}
        for g in group_probs.index:
            for target_value in [0, 1]:
                denom = float(group_probs.loc[g, target_value])
                numer = float(global_probs[target_value])
                ratio[(g, target_value)] = numer / max(denom, 1e-9)

        factors = np.array([
            ratio.get((group_key.iloc[i], int(y[i])), 1.0)
            for i in range(n)
        ])

        factors = np.power(factors, lambda_fair)
        weights *= factors

    weights = np.nan_to_num(weights, nan=1.0, posinf=cap, neginf=1.0)
    weights = np.clip(weights, 1.0 / cap, cap)

    if weights.sum() <= 0:
        weights = np.ones(n, dtype=float)

    weights = weights / weights.sum()
    return weights


def generate_gdt_resampler(
    train_df,
    n_samples,
    seed,
    sensitive_cols=None,
    fair_mode="none",
    alpha=2.0,
    lambda_fair=1.0,
    jitter_numeric=True,
    n_estimators=48,
    min_samples_leaf=8,
):
    rng = np.random.default_rng(seed)

    train = train_df.reset_index(drop=True).copy()
    sensitive_cols = sensitive_cols or []

    leaf_model = build_tree_leaf_buckets(
        train,
        seed=seed,
        n_estimators=n_estimators,
        min_samples_leaf=min_samples_leaf,
    )

    row_weights = compute_row_fair_weights(
        train,
        sensitive_cols=sensitive_cols,
        mode=fair_mode,
        alpha=alpha,
        lambda_fair=lambda_fair,
        cap=30.0,
    )

    keys = leaf_model["keys"]
    sizes = leaf_model["sizes"]
    buckets = leaf_model["buckets"]

    selected_indices = []

    for _ in range(n_samples):
        k_idx = rng.choice(len(keys), p=sizes)
        key = keys[k_idx]
        idx = buckets[key]

        if len(idx) == 1:
            selected_indices.append(int(idx[0]))
        else:
            local_weights = row_weights[idx].astype(float)
            if local_weights.sum() <= 0 or np.isnan(local_weights.sum()):
                local_weights = np.ones(len(idx)) / len(idx)
            else:
                local_weights = local_weights / local_weights.sum()

            selected_indices.append(int(rng.choice(idx, p=local_weights)))

    synth = train.iloc[selected_indices].copy().reset_index(drop=True)

    if jitter_numeric:
        numeric_cols = [
            c for c in synth.columns
            if c != "target" and pd.api.types.is_numeric_dtype(train[c])
        ]

        for c in numeric_cols:
            real_vals = pd.to_numeric(train[c], errors="coerce").dropna()
            if len(real_vals) < 10:
                continue

            std = float(real_vals.std())
            if std <= 1e-12 or np.isnan(std):
                continue

            noise = rng.normal(0, 0.01 * std, size=len(synth))
            vals = pd.to_numeric(synth[c], errors="coerce").values.astype(float) + noise

            vals = np.clip(vals, real_vals.min(), real_vals.max())

            is_int_like = np.allclose(real_vals.values, np.round(real_vals.values), atol=1e-8)
            if is_int_like:
                vals = np.round(vals).astype(int)

            synth[c] = vals

    synth = repair_synthetic_dtypes(train_df, synth)
    return synth.reset_index(drop=True)


def generate_method(train_df, method, n_samples, seed, sensitive_cols, lambda_fair=1.0):
    if method == "bootstrap":
        return generate_bootstrap(train_df, n_samples, seed)

    if method == "vanilla_gdt":
        return generate_gdt_resampler(
            train_df,
            n_samples=n_samples,
            seed=seed,
            sensitive_cols=sensitive_cols,
            fair_mode="none",
            alpha=2.0,
            lambda_fair=0.0,
        )

    if method == "single_fair_gdt":
        return generate_gdt_resampler(
            train_df,
            n_samples=n_samples,
            seed=seed,
            sensitive_cols=sensitive_cols,
            fair_mode="single_only",
            alpha=2.0,
            lambda_fair=lambda_fair,
        )

    if method == "our_multi_fair_gdt":
        return generate_gdt_resampler(
            train_df,
            n_samples=n_samples,
            seed=seed,
            sensitive_cols=sensitive_cols,
            fair_mode="multi_intersection",
            alpha=2.0,
            lambda_fair=lambda_fair,
        )

    raise ValueError(f"Unknown method: {method}")


# ============================================================
# 6. SDV generators
# ============================================================

def ensure_sdv_available():
    try:
        import sdv
        return True, None
    except Exception as e:
        return False, str(e)


def fit_sample_sdv(train_df, method, n_samples, seed):
    """
    CPU SDV baseline. Uses modern SDV if available.
    If API differs, returns clear error.
    """
    ok, err = ensure_sdv_available()
    if not ok:
        raise ImportError(
            "SDV is not installed. Install with: pip install sdv. "
            f"Original import error: {err}"
        )

    np.random.seed(seed)

    train = train_df.copy()
    train = train.reset_index(drop=True)

    # SDV expects clean column names and simple dtypes.
    for c in train.columns:
        if train[c].dtype == object:
            train[c] = train[c].astype(str).fillna("NA")

    try:
        from sdv.metadata import SingleTableMetadata
        from sdv.single_table import CTGANSynthesizer, TVAESynthesizer

        metadata = SingleTableMetadata()
        metadata.detect_from_dataframe(train)

        if method == "ctgan":
            synth = CTGANSynthesizer(
                metadata,
                epochs=100,
                verbose=False,
                cuda=False,
            )
        elif method == "tvae":
            synth = TVAESynthesizer(
                metadata,
                epochs=100,
                cuda=False,
            )
        else:
            raise ValueError(method)

        synth.fit(train)
        out = synth.sample(num_rows=n_samples)

    except Exception as e1:
        # Older SDV fallback.
        try:
            from sdv.tabular import CTGAN, TVAE

            if method == "ctgan":
                synth = CTGAN(
                    epochs=100,
                    cuda=False,
                    verbose=False,
                )
            elif method == "tvae":
                synth = TVAE(
                    epochs=100,
                    cuda=False,
                )
            else:
                raise ValueError(method)

            synth.fit(train)
            out = synth.sample(n_samples)

        except Exception as e2:
            raise RuntimeError(
                f"SDV failed. New API error: {e1}. Old API error: {e2}"
            )

    out = repair_synthetic_dtypes(train_df, out)
    return out.reset_index(drop=True)


# ============================================================
# 7. Evaluation
# ============================================================

def evaluate_downstream(
    train_synth,
    train_real,
    test_real,
    sensitive_cols,
    seed,
    min_group_count=20,
):
    feature_cols = get_feature_columns(train_real, sensitive_cols)
    feature_cols = [c for c in feature_cols if c in train_synth.columns and c in test_real.columns]

    X_train = train_synth[feature_cols].copy()
    y_train = train_synth["target"].astype(int).values

    X_test = test_real[feature_cols].copy()
    y_test = test_real["target"].astype(int).values

    models = make_downstream_models(seed)

    all_metrics = {}

    for model_name, model in models.items():
        pre = make_preprocessor(train_real, feature_cols)

        pipe = Pipeline([
            ("pre", pre),
            ("clf", model),
        ])

        try:
            pipe.fit(X_train, y_train)
            y_pred = pipe.predict(X_test).astype(int)

            try:
                y_prob = pipe.predict_proba(X_test)[:, 1]
            except Exception:
                y_prob = y_pred

            metrics = {
                f"{model_name}_accuracy": safe_float(accuracy_score(y_test, y_pred)),
                f"{model_name}_balanced_accuracy": safe_float(balanced_accuracy_score(y_test, y_pred)),
                f"{model_name}_f1": safe_float(f1_score(y_test, y_pred, zero_division=0)),
            }

            try:
                metrics[f"{model_name}_roc_auc"] = safe_float(roc_auc_score(y_test, y_prob))
            except Exception:
                metrics[f"{model_name}_roc_auc"] = None

            fair = compute_fairness_metrics(
                df_eval=test_real,
                y_true=y_test,
                y_pred=y_pred,
                sensitive_cols=sensitive_cols,
                min_group_count=min_group_count,
            )

            for k, v in fair.items():
                metrics[f"{model_name}_{k}"] = v

            all_metrics.update(metrics)

        except Exception as e:
            all_metrics[f"{model_name}_error"] = str(e)

    for key in [
        "accuracy",
        "balanced_accuracy",
        "f1",
        "roc_auc",
        "dp_diff",
        "dp_max_gap",
        "equal_opp_diff",
        "equalized_odds_diff",
        "avg_odds_diff",
    ]:
        rf_key = f"rf_{key}"
        lr_key = f"logreg_{key}"

        if rf_key in all_metrics:
            all_metrics[f"main_{key}"] = all_metrics[rf_key]
        elif lr_key in all_metrics:
            all_metrics[f"main_{key}"] = all_metrics[lr_key]
        else:
            all_metrics[f"main_{key}"] = None

    return all_metrics


def evaluate_synthetic_run(
    dataset_name,
    train_real,
    test_real,
    synth_train,
    sensitive_cols,
    seed,
    min_group_count,
):
    synth_train = clean_dataframe(synth_train)
    synth_train = add_derived_columns(synth_train, dataset_name)

    if synth_train["target"].nunique() < 2:
        extra = train_real.sample(n=len(synth_train), replace=True, random_state=seed)
        synth_train["target"] = extra["target"].values

    downstream = evaluate_downstream(
        train_synth=synth_train,
        train_real=train_real,
        test_real=test_real,
        sensitive_cols=sensitive_cols,
        seed=seed,
        min_group_count=min_group_count,
    )

    feature_cols = get_feature_columns(train_real, sensitive_cols)
    feature_cols = [c for c in feature_cols if c in synth_train.columns]

    quality = compute_quality_metrics(
        real_train=train_real,
        synth_train=synth_train,
        feature_cols=feature_cols,
        seed=seed,
    )

    result = {}
    result.update(downstream)
    result.update(quality)
    return result


def split_real_dataset(df, seed):
    y = df["target"].astype(int)
    stratify = y if y.value_counts().min() >= 2 else None

    train_real, test_real = train_test_split(
        df,
        test_size=0.30,
        random_state=seed,
        stratify=stratify,
    )

    return train_real.reset_index(drop=True), test_real.reset_index(drop=True)


# ============================================================
# 8. Task runners
# ============================================================

def run_lambda_sweep_task(task):
    out_path = Path(task["out_path"])
    if out_path.exists():
        return {"status": "skipped_exists", "out_path": str(out_path)}

    start_total = time.time()

    try:
        dataset = task["dataset"]
        seed = int(task["seed"])
        lambda_fair = float(task["lambda_fair"])
        setting = "intersectional"
        method = "our_multi_fair_gdt"

        df = load_dataset(dataset)
        sensitive_cols = SENSITIVE_SETTINGS[dataset][setting]
        df = df.dropna(subset=sensitive_cols + ["target"]).copy()

        train_real, test_real = split_real_dataset(df, seed)

        n_synth = max(50, int(round(len(train_real) * SYNTH_SIZE_RATIO)))

        start_gen = time.time()
        synth_train = generate_method(
            train_real,
            method=method,
            n_samples=n_synth,
            seed=seed,
            sensitive_cols=sensitive_cols,
            lambda_fair=lambda_fair,
        )
        gen_time = time.time() - start_gen

        start_eval = time.time()
        metrics = evaluate_synthetic_run(
            dataset_name=dataset,
            train_real=train_real,
            test_real=test_real,
            synth_train=synth_train,
            sensitive_cols=sensitive_cols,
            seed=seed,
            min_group_count=DEFAULT_MIN_GROUP_COUNT,
        )
        eval_time = time.time() - start_eval

        result = {
            "status": "ok",
            "series": "lambda_sweep",
            "dataset": dataset,
            "setting": setting,
            "method": method,
            "seed": seed,
            "lambda_fair": lambda_fair,
            "sensitive_cols": sensitive_cols,
            "n_rows_total": int(len(df)),
            "n_train_real": int(len(train_real)),
            "n_test_real": int(len(test_real)),
            "n_synth": int(len(synth_train)),
            "generation_time_sec": safe_float(gen_time),
            "evaluation_time_sec": safe_float(eval_time),
            "total_time_sec": safe_float(time.time() - start_total),
        }
        result.update(metrics)

    except Exception as e:
        result = {
            "status": "error",
            "series": "lambda_sweep",
            "error": str(e),
            "traceback": traceback.format_exc(),
            **{k: task.get(k) for k in task if k != "out_path"},
        }

    tmp = out_path.with_suffix(".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    tmp.replace(out_path)

    return {"status": result.get("status"), "out_path": str(out_path)}


def run_support_sensitivity_task(task):
    out_path = Path(task["out_path"])
    if out_path.exists():
        return {"status": "skipped_exists", "out_path": str(out_path)}

    start_total = time.time()

    try:
        dataset = task["dataset"]
        seed = int(task["seed"])
        support_threshold = int(task["support_threshold"])
        setting = "intersectional"
        method = "our_multi_fair_gdt"

        df = load_dataset(dataset)
        sensitive_cols = SENSITIVE_SETTINGS[dataset][setting]
        df = df.dropna(subset=sensitive_cols + ["target"]).copy()

        train_real, test_real = split_real_dataset(df, seed)
        n_synth = max(50, int(round(len(train_real) * SYNTH_SIZE_RATIO)))

        start_gen = time.time()
        synth_train = generate_method(
            train_real,
            method=method,
            n_samples=n_synth,
            seed=seed,
            sensitive_cols=sensitive_cols,
            lambda_fair=1.0,
        )
        gen_time = time.time() - start_gen

        start_eval = time.time()
        metrics = evaluate_synthetic_run(
            dataset_name=dataset,
            train_real=train_real,
            test_real=test_real,
            synth_train=synth_train,
            sensitive_cols=sensitive_cols,
            seed=seed,
            min_group_count=support_threshold,
        )
        eval_time = time.time() - start_eval

        result = {
            "status": "ok",
            "series": "support_sensitivity",
            "dataset": dataset,
            "setting": setting,
            "method": method,
            "seed": seed,
            "support_threshold": support_threshold,
            "sensitive_cols": sensitive_cols,
            "n_rows_total": int(len(df)),
            "n_train_real": int(len(train_real)),
            "n_test_real": int(len(test_real)),
            "n_synth": int(len(synth_train)),
            "generation_time_sec": safe_float(gen_time),
            "evaluation_time_sec": safe_float(eval_time),
            "total_time_sec": safe_float(time.time() - start_total),
        }
        result.update(metrics)

    except Exception as e:
        result = {
            "status": "error",
            "series": "support_sensitivity",
            "error": str(e),
            "traceback": traceback.format_exc(),
            **{k: task.get(k) for k in task if k != "out_path"},
        }

    tmp = out_path.with_suffix(".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    tmp.replace(out_path)

    return {"status": result.get("status"), "out_path": str(out_path)}


def run_intersectional_stress_task(task):
    out_path = Path(task["out_path"])
    if out_path.exists():
        return {"status": "skipped_exists", "out_path": str(out_path)}

    start_total = time.time()

    try:
        dataset = task["dataset"]
        seed = int(task["seed"])
        method = task["method"]
        stress_setting = task["stress_setting"]
        sensitive_cols = task["sensitive_cols"]

        df = load_dataset(dataset)
        df = df.dropna(subset=sensitive_cols + ["target"]).copy()

        train_real, test_real = split_real_dataset(df, seed)
        n_synth = max(50, int(round(len(train_real) * SYNTH_SIZE_RATIO)))

        start_gen = time.time()
        synth_train = generate_method(
            train_real,
            method=method,
            n_samples=n_synth,
            seed=seed,
            sensitive_cols=sensitive_cols,
            lambda_fair=1.0,
        )
        gen_time = time.time() - start_gen

        start_eval = time.time()
        metrics = evaluate_synthetic_run(
            dataset_name=dataset,
            train_real=train_real,
            test_real=test_real,
            synth_train=synth_train,
            sensitive_cols=sensitive_cols,
            seed=seed,
            min_group_count=DEFAULT_MIN_GROUP_COUNT,
        )
        eval_time = time.time() - start_eval

        result = {
            "status": "ok",
            "series": "intersectional_stress",
            "dataset": dataset,
            "stress_setting": stress_setting,
            "method": method,
            "seed": seed,
            "sensitive_cols": sensitive_cols,
            "n_sensitive_attrs": int(len(sensitive_cols)),
            "n_rows_total": int(len(df)),
            "n_train_real": int(len(train_real)),
            "n_test_real": int(len(test_real)),
            "n_synth": int(len(synth_train)),
            "generation_time_sec": safe_float(gen_time),
            "evaluation_time_sec": safe_float(eval_time),
            "total_time_sec": safe_float(time.time() - start_total),
        }
        result.update(metrics)

    except Exception as e:
        result = {
            "status": "error",
            "series": "intersectional_stress",
            "error": str(e),
            "traceback": traceback.format_exc(),
            **{k: task.get(k) for k in task if k != "out_path"},
        }

    tmp = out_path.with_suffix(".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    tmp.replace(out_path)

    return {"status": result.get("status"), "out_path": str(out_path)}


def run_sdv_task(task):
    out_path = Path(task["out_path"])
    if out_path.exists():
        return {"status": "skipped_exists", "out_path": str(out_path)}

    start_total = time.time()

    try:
        if not ENABLE_SDV_BASELINES:
            raise RuntimeError("ENABLE_SDV_BASELINES=False")

        dataset = task["dataset"]
        seed = int(task["seed"])
        setting = task["setting"]
        method = task["method"]

        df = load_dataset(dataset)
        sensitive_cols = SENSITIVE_SETTINGS[dataset][setting]
        df = df.dropna(subset=sensitive_cols + ["target"]).copy()

        train_real, test_real = split_real_dataset(df, seed)
        n_synth = max(50, int(round(len(train_real) * SYNTH_SIZE_RATIO)))

        start_gen = time.time()
        synth_train = fit_sample_sdv(
            train_df=train_real,
            method=method,
            n_samples=n_synth,
            seed=seed,
        )
        gen_time = time.time() - start_gen

        start_eval = time.time()
        metrics = evaluate_synthetic_run(
            dataset_name=dataset,
            train_real=train_real,
            test_real=test_real,
            synth_train=synth_train,
            sensitive_cols=sensitive_cols,
            seed=seed,
            min_group_count=DEFAULT_MIN_GROUP_COUNT,
        )
        eval_time = time.time() - start_eval

        result = {
            "status": "ok",
            "series": "sdv_baselines",
            "dataset": dataset,
            "setting": setting,
            "method": method,
            "seed": seed,
            "sensitive_cols": sensitive_cols,
            "n_rows_total": int(len(df)),
            "n_train_real": int(len(train_real)),
            "n_test_real": int(len(test_real)),
            "n_synth": int(len(synth_train)),
            "generation_time_sec": safe_float(gen_time),
            "evaluation_time_sec": safe_float(eval_time),
            "total_time_sec": safe_float(time.time() - start_total),
        }
        result.update(metrics)

    except Exception as e:
        result = {
            "status": "error",
            "series": "sdv_baselines",
            "error": str(e),
            "traceback": traceback.format_exc(),
            **{k: task.get(k) for k in task if k != "out_path"},
        }

    tmp = out_path.with_suffix(".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    tmp.replace(out_path)

    return {"status": result.get("status"), "out_path": str(out_path)}


def run_runtime_scaling_task(task):
    out_path = Path(task["out_path"])
    if out_path.exists():
        return {"status": "skipped_exists", "out_path": str(out_path)}

    start_total = time.time()

    try:
        dataset = task["dataset"]
        seed = int(task["seed"])
        setting = task["setting"]
        method = task["method"]
        train_size = task["train_size"]

        df = load_dataset(dataset)
        sensitive_cols = SENSITIVE_SETTINGS[dataset][setting]
        df = df.dropna(subset=sensitive_cols + ["target"]).copy()

        train_real_full, test_real = split_real_dataset(df, seed)

        if train_size == "full":
            train_real = train_real_full.copy()
            actual_train_size = len(train_real)
        else:
            train_size_int = int(train_size)
            actual_train_size = min(train_size_int, len(train_real_full))
            train_real = train_real_full.sample(
                n=actual_train_size,
                replace=False,
                random_state=seed,
            ).reset_index(drop=True)

        n_synth = max(50, int(round(len(train_real) * SYNTH_SIZE_RATIO)))

        start_gen = time.time()
        synth_train = generate_method(
            train_real,
            method=method,
            n_samples=n_synth,
            seed=seed,
            sensitive_cols=sensitive_cols,
            lambda_fair=1.0,
        )
        gen_time = time.time() - start_gen

        start_eval = time.time()
        metrics = evaluate_synthetic_run(
            dataset_name=dataset,
            train_real=train_real,
            test_real=test_real,
            synth_train=synth_train,
            sensitive_cols=sensitive_cols,
            seed=seed,
            min_group_count=DEFAULT_MIN_GROUP_COUNT,
        )
        eval_time = time.time() - start_eval

        result = {
            "status": "ok",
            "series": "runtime_scaling",
            "dataset": dataset,
            "setting": setting,
            "method": method,
            "seed": seed,
            "train_size_requested": train_size,
            "train_size_actual": int(actual_train_size),
            "sensitive_cols": sensitive_cols,
            "n_rows_total": int(len(df)),
            "n_train_real": int(len(train_real)),
            "n_test_real": int(len(test_real)),
            "n_synth": int(len(synth_train)),
            "generation_time_sec": safe_float(gen_time),
            "evaluation_time_sec": safe_float(eval_time),
            "total_time_sec": safe_float(time.time() - start_total),
        }
        result.update(metrics)

    except Exception as e:
        result = {
            "status": "error",
            "series": "runtime_scaling",
            "error": str(e),
            "traceback": traceback.format_exc(),
            **{k: task.get(k) for k in task if k != "out_path"},
        }

    tmp = out_path.with_suffix(".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    tmp.replace(out_path)

    return {"status": result.get("status"), "out_path": str(out_path)}


# ============================================================
# 9. Task construction
# ============================================================

def make_id(*parts):
    return "__".join(str(p).replace("/", "_").replace(" ", "_") for p in parts)


def build_lambda_tasks():
    tasks = []
    for dataset in LAMBDA_DATASETS:
        for lam in LAMBDA_VALUES:
            for seed in SEEDS_5:
                run_id = make_id("lambda", dataset, "intersectional", f"lambda{lam}", f"seed{seed}")
                out_path = SERIES_DIRS["lambda_sweep"] / f"{run_id}.json"
                tasks.append({
                    "series": "lambda_sweep",
                    "dataset": dataset,
                    "lambda_fair": lam,
                    "seed": seed,
                    "out_path": str(out_path),
                })
    return tasks


def build_support_tasks():
    tasks = []
    for dataset in SUPPORT_DATASETS:
        for thr in SUPPORT_THRESHOLDS:
            for seed in SEEDS_5:
                run_id = make_id("support", dataset, "intersectional", f"thr{thr}", f"seed{seed}")
                out_path = SERIES_DIRS["support_sensitivity"] / f"{run_id}.json"
                tasks.append({
                    "series": "support_sensitivity",
                    "dataset": dataset,
                    "support_threshold": thr,
                    "seed": seed,
                    "out_path": str(out_path),
                })
    return tasks


def build_stress_tasks():
    tasks = []
    for dataset in STRESS_DATASETS:
        configs = STRESS_SENSITIVE_CONFIGS[dataset]
        for stress_setting, sensitive_cols in configs.items():
            for method in STRESS_METHODS:
                for seed in SEEDS_5:
                    run_id = make_id("stress", dataset, stress_setting, method, f"seed{seed}")
                    out_path = SERIES_DIRS["intersectional_stress"] / f"{run_id}.json"
                    tasks.append({
                        "series": "intersectional_stress",
                        "dataset": dataset,
                        "stress_setting": stress_setting,
                        "sensitive_cols": sensitive_cols,
                        "method": method,
                        "seed": seed,
                        "out_path": str(out_path),
                    })
    return tasks


def build_sdv_tasks():
    tasks = []
    for dataset in SDV_DATASETS:
        for setting in SDV_SETTINGS:
            for method in SDV_METHODS:
                for seed in SEEDS_3:
                    run_id = make_id("sdv", dataset, setting, method, f"seed{seed}")
                    out_path = SERIES_DIRS["sdv_baselines"] / f"{run_id}.json"
                    tasks.append({
                        "series": "sdv_baselines",
                        "dataset": dataset,
                        "setting": setting,
                        "method": method,
                        "seed": seed,
                        "out_path": str(out_path),
                    })
    return tasks


def build_runtime_tasks():
    tasks = []
    for dataset in RUNTIME_DATASETS:
        for train_size in RUNTIME_TRAIN_SIZES:
            for setting in RUNTIME_SETTINGS:
                for method in RUNTIME_METHODS:
                    for seed in SEEDS_3:
                        run_id = make_id("runtime", dataset, f"n{train_size}", setting, method, f"seed{seed}")
                        out_path = SERIES_DIRS["runtime_scaling"] / f"{run_id}.json"
                        tasks.append({
                            "series": "runtime_scaling",
                            "dataset": dataset,
                            "train_size": train_size,
                            "setting": setting,
                            "method": method,
                            "seed": seed,
                            "out_path": str(out_path),
                        })
    return tasks


def build_all_tasks():
    tasks = []
    tasks += build_lambda_tasks()
    tasks += build_support_tasks()
    tasks += build_stress_tasks()
    tasks += build_sdv_tasks()
    tasks += build_runtime_tasks()
    return tasks


# ============================================================
# 10. Aggregation and tables
# ============================================================

def read_json_safe(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        return {
            "status": "bad_json",
            "path": str(path),
            "error": str(e),
        }


def aggregate_current_results():
    rows = []
    for p in sorted(RUNS_DIR.rglob("*.json")):
        rows.append(read_json_safe(p))

    df = pd.DataFrame(rows)

    all_path = RESULTS_DIR / "extra_all_results.csv"
    df.to_csv(all_path, index=False)

    if len(df) == 0:
        return df

    for series in sorted(df["series"].dropna().unique()) if "series" in df.columns else []:
        sub = df[df["series"] == series].copy()
        sub.to_csv(RESULTS_DIR / f"{series}_results.csv", index=False)

    ok = df[df["status"] == "ok"].copy() if "status" in df.columns else df.copy()

    metric_cols = [
        "main_roc_auc",
        "main_f1",
        "main_balanced_accuracy",
        "main_dp_diff",
        "main_equal_opp_diff",
        "main_equalized_odds_diff",
        "main_avg_odds_diff",
        "quality_numeric_ks_mean",
        "quality_categorical_tvd_mean",
        "quality_corr_distance",
        "quality_detection_auc",
        "generation_time_sec",
        "evaluation_time_sec",
        "total_time_sec",
        "rf_n_groups_total",
        "rf_n_groups_valid",
        "rf_min_group_count",
        "rf_max_group_count",
    ]

    metric_cols = [c for c in metric_cols if c in ok.columns]

    # Series-specific summaries.
    summary_specs = {
        "lambda_sweep": ["series", "dataset", "lambda_fair"],
        "support_sensitivity": ["series", "dataset", "support_threshold"],
        "intersectional_stress": ["series", "dataset", "stress_setting", "method"],
        "sdv_baselines": ["series", "dataset", "setting", "method"],
        "runtime_scaling": ["series", "dataset", "train_size_requested", "setting", "method"],
    }

    for series, group_cols in summary_specs.items():
        sub = ok[ok["series"] == series].copy() if "series" in ok.columns else pd.DataFrame()
        if len(sub) == 0:
            continue

        group_cols = [c for c in group_cols if c in sub.columns]
        if not group_cols or not metric_cols:
            continue

        existing_metrics = [c for c in metric_cols if c in sub.columns]
        if not existing_metrics:
            continue

        summary = (
            sub.groupby(group_cols)[existing_metrics]
            .agg(["mean", "std", "count"])
            .reset_index()
        )

        summary.columns = [
            "_".join([str(x) for x in col if str(x) != ""])
            if isinstance(col, tuple) else str(col)
            for col in summary.columns
        ]

        summary.to_csv(RESULTS_DIR / f"{series}_summary_mean_std.csv", index=False)

    # Status table.
    status_cols = ["series", "dataset", "setting", "stress_setting", "method", "status"]
    status_cols = [c for c in status_cols if c in df.columns]
    if status_cols:
        status = df.groupby(status_cols, dropna=False).size().reset_index(name="count")
        status.to_csv(RESULTS_DIR / "extra_run_status.csv", index=False)

    # Combined quality-vs-fairness dataset:
    build_quality_vs_fairness_table(df)

    return df


def build_quality_vs_fairness_table(extra_df):
    rows = []

    if PREV_ALL_RESULTS.exists():
        try:
            prev = pd.read_csv(PREV_ALL_RESULTS)
            prev["source"] = "main_1020"
            rows.append(prev)
        except Exception as e:
            log(f"[WARN] Could not read previous all_results: {e}")

    if extra_df is not None and len(extra_df) > 0:
        extra = extra_df.copy()
        extra["source"] = "extra_series"
        rows.append(extra)

    if not rows:
        return

    combined = pd.concat(rows, axis=0, ignore_index=True, sort=False)

    wanted = [
        "source",
        "series",
        "run_type",
        "dataset",
        "setting",
        "stress_setting",
        "method",
        "seed",
        "lambda_fair",
        "support_threshold",
        "train_size_requested",
        "main_roc_auc",
        "main_f1",
        "main_balanced_accuracy",
        "main_dp_diff",
        "main_equal_opp_diff",
        "main_equalized_odds_diff",
        "main_avg_odds_diff",
        "quality_numeric_ks_mean",
        "quality_categorical_tvd_mean",
        "quality_corr_distance",
        "quality_detection_auc",
        "generation_time_sec",
        "total_time_sec",
        "status",
    ]

    existing = [c for c in wanted if c in combined.columns]
    out = combined[existing].copy()

    out.to_csv(RESULTS_DIR / "quality_vs_fairness_all_sources.csv", index=False)


# ============================================================
# 11. Plotting
# ============================================================

def make_plots():
    import matplotlib.pyplot as plt

    plot_dir = RESULTS_DIR / "plots"
    plot_dir.mkdir(parents=True, exist_ok=True)

    all_path = RESULTS_DIR / "extra_all_results.csv"
    if not all_path.exists():
        return

    df = pd.read_csv(all_path)
    ok = df[df["status"] == "ok"].copy() if "status" in df.columns else df.copy()

    # --------------------------------------------------------
    # Lambda Pareto plots
    # --------------------------------------------------------
    lam = ok[ok["series"] == "lambda_sweep"].copy() if "series" in ok.columns else pd.DataFrame()

    if len(lam) > 0:
        g = (
            lam.groupby(["dataset", "lambda_fair"])[
                ["main_roc_auc", "main_equalized_odds_diff", "main_dp_diff", "main_avg_odds_diff"]
            ]
            .mean()
            .reset_index()
        )

        for dataset in sorted(g["dataset"].unique()):
            sub = g[g["dataset"] == dataset].sort_values("lambda_fair")

            plt.figure(figsize=(6, 4))
            plt.plot(sub["main_equalized_odds_diff"], sub["main_roc_auc"], marker="o")
            for _, r in sub.iterrows():
                plt.text(r["main_equalized_odds_diff"], r["main_roc_auc"], str(r["lambda_fair"]), fontsize=8)
            plt.xlabel("Equalized Odds Difference ↓")
            plt.ylabel("ROC-AUC ↑")
            plt.title(f"Pareto: {dataset}")
            plt.tight_layout()
            plt.savefig(plot_dir / f"pareto_lambda_{dataset}.pdf")
            plt.savefig(plot_dir / f"pareto_lambda_{dataset}.png", dpi=200)
            plt.close()

    # --------------------------------------------------------
    # Support sensitivity plots
    # --------------------------------------------------------
    sup = ok[ok["series"] == "support_sensitivity"].copy() if "series" in ok.columns else pd.DataFrame()

    if len(sup) > 0:
        g = (
            sup.groupby(["dataset", "support_threshold"])[
                ["main_equalized_odds_diff", "main_dp_diff", "rf_n_groups_valid"]
            ]
            .mean()
            .reset_index()
        )

        for dataset in sorted(g["dataset"].unique()):
            sub = g[g["dataset"] == dataset].sort_values("support_threshold")

            plt.figure(figsize=(6, 4))
            plt.plot(sub["support_threshold"], sub["main_equalized_odds_diff"], marker="o")
            plt.xlabel("Minimum group support")
            plt.ylabel("Equalized Odds Difference ↓")
            plt.title(f"Support sensitivity: {dataset}")
            plt.tight_layout()
            plt.savefig(plot_dir / f"support_sensitivity_{dataset}.pdf")
            plt.savefig(plot_dir / f"support_sensitivity_{dataset}.png", dpi=200)
            plt.close()

    # --------------------------------------------------------
    # Intersectional stress plots
    # --------------------------------------------------------
    stress = ok[ok["series"] == "intersectional_stress"].copy() if "series" in ok.columns else pd.DataFrame()

    if len(stress) > 0:
        g = (
            stress.groupby(["dataset", "stress_setting", "n_sensitive_attrs", "method"])[
                ["main_roc_auc", "main_equalized_odds_diff", "main_dp_diff"]
            ]
            .mean()
            .reset_index()
        )

        for dataset in sorted(g["dataset"].unique()):
            sub = g[g["dataset"] == dataset].copy()
            methods = sorted(sub["method"].unique())

            plt.figure(figsize=(7, 4))
            for m in methods:
                sm = sub[sub["method"] == m].sort_values("n_sensitive_attrs")
                plt.plot(sm["n_sensitive_attrs"], sm["main_equalized_odds_diff"], marker="o", label=m)
            plt.xlabel("Number of sensitive attributes")
            plt.ylabel("Equalized Odds Difference ↓")
            plt.title(f"Intersectional stress: {dataset}")
            plt.legend(fontsize=8)
            plt.tight_layout()
            plt.savefig(plot_dir / f"intersectional_stress_{dataset}.pdf")
            plt.savefig(plot_dir / f"intersectional_stress_{dataset}.png", dpi=200)
            plt.close()

    # --------------------------------------------------------
    # Runtime scaling plots
    # --------------------------------------------------------
    rt = ok[ok["series"] == "runtime_scaling"].copy() if "series" in ok.columns else pd.DataFrame()

    if len(rt) > 0:
        rt["train_size_numeric"] = rt["train_size_actual"]

        g = (
            rt.groupby(["dataset", "setting", "method", "train_size_numeric"])[
                ["generation_time_sec", "total_time_sec", "main_equalized_odds_diff", "main_roc_auc"]
            ]
            .mean()
            .reset_index()
        )

        for setting in sorted(g["setting"].unique()):
            sub = g[g["setting"] == setting]
            methods = sorted(sub["method"].unique())

            plt.figure(figsize=(7, 4))
            for m in methods:
                sm = sub[sub["method"] == m].sort_values("train_size_numeric")
                plt.plot(sm["train_size_numeric"], sm["generation_time_sec"], marker="o", label=m)
            plt.xlabel("Training size")
            plt.ylabel("Generation time, sec ↓")
            plt.title(f"Runtime scaling: {setting}")
            plt.legend(fontsize=8)
            plt.tight_layout()
            plt.savefig(plot_dir / f"runtime_scaling_{setting}.pdf")
            plt.savefig(plot_dir / f"runtime_scaling_{setting}.png", dpi=200)
            plt.close()

    # --------------------------------------------------------
    # Quality vs fairness from all sources
    # --------------------------------------------------------
    qvf_path = RESULTS_DIR / "quality_vs_fairness_all_sources.csv"
    if qvf_path.exists():
        qvf = pd.read_csv(qvf_path)
        qvf = qvf[qvf["status"] == "ok"].copy() if "status" in qvf.columns else qvf

        if "quality_detection_auc" in qvf.columns and "main_equalized_odds_diff" in qvf.columns:
            sub = qvf.dropna(subset=["quality_detection_auc", "main_equalized_odds_diff"])

            if len(sub) > 0:
                plt.figure(figsize=(6, 4))
                for method in sorted(sub["method"].dropna().unique()):
                    sm = sub[sub["method"] == method]
                    if len(sm) == 0:
                        continue
                    plt.scatter(
                        sm["main_equalized_odds_diff"],
                        sm["quality_detection_auc"],
                        label=method,
                        alpha=0.55,
                        s=16,
                    )
                plt.xlabel("Equalized Odds Difference ↓")
                plt.ylabel("Detection AUC, real vs synthetic")
                plt.title("Synthetic quality vs fairness")
                plt.legend(fontsize=7)
                plt.tight_layout()
                plt.savefig(plot_dir / "quality_vs_fairness_detection_auc.pdf")
                plt.savefig(plot_dir / "quality_vs_fairness_detection_auc.png", dpi=200)
                plt.close()

    log(f"[PLOTS] saved to {plot_dir}")


# ============================================================
# 12. LaTeX table snippets
# ============================================================

def mean_std_str(mean, std, digits=3):
    if pd.isna(mean):
        return "--"
    if pd.isna(std):
        return f"{mean:.{digits}f}"
    return f"{mean:.{digits}f} $\\pm$ {std:.{digits}f}"


def create_latex_tables():
    table_dir = RESULTS_DIR / "latex_tables"
    table_dir.mkdir(parents=True, exist_ok=True)

    # Lambda sweep summary.
    path = RESULTS_DIR / "lambda_sweep_summary_mean_std.csv"
    if path.exists():
        df = pd.read_csv(path)

        cols = [
            "dataset",
            "lambda_fair",
            "main_roc_auc_mean",
            "main_roc_auc_std",
            "main_equalized_odds_diff_mean",
            "main_equalized_odds_diff_std",
            "main_dp_diff_mean",
            "main_dp_diff_std",
            "main_avg_odds_diff_mean",
            "main_avg_odds_diff_std",
        ]

        existing = [c for c in cols if c in df.columns]
        small = df[existing].copy()

        lines = []
        lines.append("\\begin{table}[t]")
        lines.append("\\centering")
        lines.append("\\caption{Fairness--utility trade-off under different fairness penalty values.}")
        lines.append("\\label{tab:lambda_sweep}")
        lines.append("\\resizebox{\\columnwidth}{!}{")
        lines.append("\\begin{tabular}{llcccc}")
        lines.append("\\toprule")
        lines.append("Dataset & $\\lambda$ & AUC $\\uparrow$ & EO Diff. $\\downarrow$ & DP Diff. $\\downarrow$ & Avg. Odds $\\downarrow$ \\\\")
        lines.append("\\midrule")

        for _, r in small.iterrows():
            lines.append(
                f"{r.get('dataset')} & {r.get('lambda_fair')} & "
                f"{mean_std_str(r.get('main_roc_auc_mean'), r.get('main_roc_auc_std'))} & "
                f"{mean_std_str(r.get('main_equalized_odds_diff_mean'), r.get('main_equalized_odds_diff_std'))} & "
                f"{mean_std_str(r.get('main_dp_diff_mean'), r.get('main_dp_diff_std'))} & "
                f"{mean_std_str(r.get('main_avg_odds_diff_mean'), r.get('main_avg_odds_diff_std'))} \\\\"
            )

        lines.append("\\bottomrule")
        lines.append("\\end{tabular}")
        lines.append("}")
        lines.append("\\end{table}")

        (table_dir / "lambda_sweep_table.tex").write_text("\n".join(lines), encoding="utf-8")

    # Stress summary.
    path = RESULTS_DIR / "intersectional_stress_summary_mean_std.csv"
    if path.exists():
        df = pd.read_csv(path)

        lines = []
        lines.append("\\begin{table*}[t]")
        lines.append("\\centering")
        lines.append("\\caption{Intersectional stress test with increasing number of sensitive attributes.}")
        lines.append("\\label{tab:intersectional_stress}")
        lines.append("\\resizebox{\\textwidth}{!}{")
        lines.append("\\begin{tabular}{lllcccc}")
        lines.append("\\toprule")
        lines.append("Dataset & Setting & Method & AUC $\\uparrow$ & EO Diff. $\\downarrow$ & DP Diff. $\\downarrow$ & Avg. Odds $\\downarrow$ \\\\")
        lines.append("\\midrule")

        for _, r in df.iterrows():
            lines.append(
                f"{r.get('dataset')} & {r.get('stress_setting')} & {r.get('method')} & "
                f"{mean_std_str(r.get('main_roc_auc_mean'), r.get('main_roc_auc_std'))} & "
                f"{mean_std_str(r.get('main_equalized_odds_diff_mean'), r.get('main_equalized_odds_diff_std'))} & "
                f"{mean_std_str(r.get('main_dp_diff_mean'), r.get('main_dp_diff_std'))} & "
                f"{mean_std_str(r.get('main_avg_odds_diff_mean'), r.get('main_avg_odds_diff_std'))} \\\\"
            )

        lines.append("\\bottomrule")
        lines.append("\\end{tabular}")
        lines.append("}")
        lines.append("\\end{table*}")

        (table_dir / "intersectional_stress_table.tex").write_text("\n".join(lines), encoding="utf-8")

    # Runtime summary.
    path = RESULTS_DIR / "runtime_scaling_summary_mean_std.csv"
    if path.exists():
        df = pd.read_csv(path)

        lines = []
        lines.append("\\begin{table}[t]")
        lines.append("\\centering")
        lines.append("\\caption{Runtime scaling on Adult under single-attribute and intersectional settings.}")
        lines.append("\\label{tab:runtime_scaling}")
        lines.append("\\resizebox{\\columnwidth}{!}{")
        lines.append("\\begin{tabular}{llllcc}")
        lines.append("\\toprule")
        lines.append("Size & Setting & Method & Gen. Time $\\downarrow$ & AUC $\\uparrow$ & EO Diff. $\\downarrow$ \\\\")
        lines.append("\\midrule")

        for _, r in df.iterrows():
            lines.append(
                f"{r.get('train_size_requested')} & {r.get('setting')} & {r.get('method')} & "
                f"{mean_std_str(r.get('generation_time_sec_mean'), r.get('generation_time_sec_std'))} & "
                f"{mean_std_str(r.get('main_roc_auc_mean'), r.get('main_roc_auc_std'))} & "
                f"{mean_std_str(r.get('main_equalized_odds_diff_mean'), r.get('main_equalized_odds_diff_std'))} \\\\"
            )

        lines.append("\\bottomrule")
        lines.append("\\end{tabular}")
        lines.append("}")
        lines.append("\\end{table}")

        (table_dir / "runtime_scaling_table.tex").write_text("\n".join(lines), encoding="utf-8")

    log(f"[LATEX] saved to {table_dir}")


# ============================================================
# 13. Parallel execution
# ============================================================

RUNNER_BY_SERIES = {
    "lambda_sweep": run_lambda_sweep_task,
    "support_sensitivity": run_support_sensitivity_task,
    "intersectional_stress": run_intersectional_stress_task,
    "sdv_baselines": run_sdv_task,
    "runtime_scaling": run_runtime_scaling_task,
}


def run_tasks_for_series(series_name, tasks):
    if not tasks:
        log(f"[{series_name}] no tasks")
        return

    pending = [t for t in tasks if not Path(t["out_path"]).exists()]
    done = len(tasks) - len(pending)

    log("=" * 80)
    log(f"SERIES: {series_name}")
    log(f"total={len(tasks)} done={done} pending={len(pending)}")
    log("=" * 80)

    if not pending:
        return

    runner = RUNNER_BY_SERIES[series_name]

    start = time.time()
    ok_count = 0
    err_count = 0
    skip_count = 0
    completed = 0

    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futs = [ex.submit(runner, t) for t in pending]

        for fut in as_completed(futs):
            completed += 1

            try:
                res = fut.result()
                status = res.get("status", "unknown")
            except Exception as e:
                status = "executor_error"
                res = {"error": str(e)}

            if status == "ok":
                ok_count += 1
            elif status == "skipped_exists":
                skip_count += 1
            else:
                err_count += 1

            if completed % 5 == 0 or completed == len(pending):
                elapsed = time.time() - start
                rate = completed / max(elapsed, 1e-9)

                log(
                    f"[{series_name}] "
                    f"{completed}/{len(pending)} "
                    f"ok={ok_count} err={err_count} skip={skip_count} "
                    f"elapsed={elapsed/60:.1f} min rate={rate:.3f} runs/sec"
                )

            if completed % 25 == 0:
                aggregate_current_results()


# ============================================================
# 14. Main
# ============================================================

def main():
    log("=" * 80)
    log("ICDM EXTRA SERIES CPU")
    log("=" * 80)
    log(f"BASE: {BASE}")
    log(f"PROCESSED_DIR: {PROCESSED_DIR}")
    log(f"PREV_ALL_RESULTS: {PREV_ALL_RESULTS} exists={PREV_ALL_RESULTS.exists()}")
    log(f"RESULTS_DIR: {RESULTS_DIR}")
    log(f"MAX_WORKERS: {MAX_WORKERS}")
    log(f"ENABLE_SDV_BASELINES: {ENABLE_SDV_BASELINES}")
    log("=" * 80)

    log("\nDataset availability:")
    for name, fname in DATASET_FILES.items():
        p = PROCESSED_DIR / fname
        log(f"  {name:<25} {'OK' if p.exists() else 'MISSING'} {p}")

    all_tasks = build_all_tasks()

    manifest_path = RESULTS_DIR / "extra_task_manifest.json"
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(all_tasks, f, indent=2, ensure_ascii=False)

    log("\nTask counts:")
    by_series = {}
    for t in all_tasks:
        by_series[t["series"]] = by_series.get(t["series"], 0) + 1

    for k, v in by_series.items():
        log(f"  {k:<25} {v}")

    log(f"  {'TOTAL':<25} {len(all_tasks)}")
    log(f"\nSaved manifest: {manifest_path}")

    # Run in order. SDV last or near-last because it may fail if not installed.
    ordered_series = [
        "lambda_sweep",
        "support_sensitivity",
        "intersectional_stress",
        "runtime_scaling",
        "sdv_baselines",
    ]

    for series_name in ordered_series:
        tasks = [t for t in all_tasks if t["series"] == series_name]

        if series_name == "sdv_baselines" and not ENABLE_SDV_BASELINES:
            log("[SKIP] sdv_baselines because ENABLE_SDV_BASELINES=False")
            continue

        run_tasks_for_series(series_name, tasks)

        log(f"[AGGREGATE] after {series_name}")
        aggregate_current_results()

    log("\nFinal aggregation...")
    df = aggregate_current_results()

    log("\nMaking plots...")
    try:
        make_plots()
    except Exception as e:
        log(f"[WARN] plotting failed: {e}")
        log(traceback.format_exc())

    log("\nMaking LaTeX tables...")
    try:
        create_latex_tables()
    except Exception as e:
        log(f"[WARN] LaTeX table creation failed: {e}")
        log(traceback.format_exc())

    log("=" * 80)
    log("DONE")
    log("=" * 80)
    log(f"Rows aggregated: {len(df)}")
    log(f"Extra all results: {RESULTS_DIR / 'extra_all_results.csv'}")
    log(f"Quality vs fairness: {RESULTS_DIR / 'quality_vs_fairness_all_sources.csv'}")
    log(f"Status: {RESULTS_DIR / 'extra_run_status.csv'}")
    log(f"Plots: {RESULTS_DIR / 'plots'}")
    log(f"LaTeX tables: {RESULTS_DIR / 'latex_tables'}")
    log("=" * 80)


if __name__ == "__main__":
    main()


ICDM EXTRA SERIES CPU
BASE: /home/tahiti/DataGenaration
PROCESSED_DIR: /home/tahiti/DataGenaration/fairness_datasets/processed
PREV_ALL_RESULTS: /home/tahiti/DataGenaration/RESULTS/icdm_fairness_final_cpu/all_results.csv exists=True
RESULTS_DIR: /home/tahiti/DataGenaration/RESULTS/icdm_fairness_extra_series_cpu
MAX_WORKERS: 10
ENABLE_SDV_BASELINES: True

Dataset availability:
  adult                     OK /home/tahiti/DataGenaration/fairness_datasets/processed/adult_processed.csv
  compas                    OK /home/tahiti/DataGenaration/fairness_datasets/processed/compas_processed.csv
  german_credit             OK /home/tahiti/DataGenaration/fairness_datasets/processed/german_credit_processed.csv
  bank_marketing            OK /home/tahiti/DataGenaration/fairness_datasets/processed/bank_marketing_processed.csv
  default_credit_card       OK /home/tahiti/DataGenaration/fairness_datasets/processed/default_credit_card_processed.csv
  communities_crime         OK /home/tahiti/DataGenara

[intersectional_stress] 150/225 ok=150 err=0 skip=0 elapsed=5.9 min rate=0.423 runs/sec
[intersectional_stress] 155/225 ok=155 err=0 skip=0 elapsed=5.9 min rate=0.436 runs/sec
[intersectional_stress] 160/225 ok=160 err=0 skip=0 elapsed=6.3 min rate=0.425 runs/sec
[intersectional_stress] 165/225 ok=165 err=0 skip=0 elapsed=6.3 min rate=0.436 runs/sec
[intersectional_stress] 170/225 ok=170 err=0 skip=0 elapsed=6.8 min rate=0.420 runs/sec
[intersectional_stress] 175/225 ok=175 err=0 skip=0 elapsed=6.8 min rate=0.430 runs/sec
[intersectional_stress] 180/225 ok=180 err=0 skip=0 elapsed=7.2 min rate=0.415 runs/sec
[intersectional_stress] 185/225 ok=185 err=0 skip=0 elapsed=7.2 min rate=0.426 runs/sec
[intersectional_stress] 190/225 ok=190 err=0 skip=0 elapsed=7.7 min rate=0.411 runs/sec
[intersectional_stress] 195/225 ok=195 err=0 skip=0 elapsed=7.7 min rate=0.421 runs/sec
[intersectional_stress] 200/225 ok=200 err=0 skip=0 elapsed=8.2 min rate=0.408 runs/sec
[intersectional_stress] 205/225 